In [26]:
import torch
import torch.nn.functional as F
torch.set_printoptions(precision=2)

T = 5
d = 3
W = 2
tokens = ["A", "B", "C", "D", "E"]

X = torch.tensor([
    [1, 0, 2],
    [0, 1, 1],
    [2, 1, 0],
    [1, 2, 1],
    [0, 0, 2],
], dtype=torch.float)

Wq1 = torch.eye(d)
Wk1 = torch.tensor([[0,1,0],[1,0,0],[0,0,1]], dtype=torch.float)
Wv1 = torch.tensor([[1,0,0],[0,0,1],[0,1,0]], dtype=torch.float)

Wq2 = torch.tensor([[1,0,1],[0,1,0],[1,0,0]], dtype=torch.float)
Wk2 = torch.tensor([[0,0,1],[1,1,0],[0,1,0]], dtype=torch.float)
Wv2 = torch.tensor([[0,1,0],[1,0,0],[0,0,1]], dtype=torch.float)

# Simple FFN: linear up -> ReLU -> linear back down (same dim for simplicity)
Wff1_up   = torch.ones(d, d) * 0.5
Wff1_down = torch.ones(d, d) * 0.5
Wff2_up   = torch.ones(d, d) * 0.5
Wff2_down = torch.ones(d, d) * 0.5


def print_matrix(label, mat, row_labels):
    print(f"\n── {label}  (shape {list(mat.shape)}) ──")
    print(f"  {'':4}", [f"d{i}" for i in range(mat.shape[1])])
    for i, tok in enumerate(row_labels):
        print(f"  {tok}   {[round(x, 2) for x in mat[i].tolist()]}")


def build_swa_mask(T, W):
    mask = torch.full((T, T), float('-inf'))
    for i in range(T):
        start = max(0, i - W + 1)
        mask[i, start:i+1] = 0.0
    return mask


def swa_attention(X, Wq, Wk, Wv, W):
    """Just the attention sublayer — returns attn output (same shape as X)."""
    scale = X.shape[1] ** 0.5
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv

    print_matrix("Q  (X @ Wq)", Q, tokens)
    print_matrix("K  (X @ Wk)", K, tokens)
    print_matrix("V  (X @ Wv)", V, tokens)

    scores_raw = (Q @ K.T) / scale
    mask = build_swa_mask(T, W)

    print(f"\n── SWA Mask  (W={W}, causal) ──")
    print(f"  query↓  key→  {tokens}")
    for i, tok in enumerate(tokens):
        row = ["  ✓" if mask[i, j] == 0 else "  ✗" for j in range(T)]
        print(f"  {tok}          {''.join(row)}")

    scores_masked = scores_raw + mask
    print_matrix("Masked Scores", scores_masked, tokens)

    attn_weights = torch.softmax(scores_masked, dim=-1)
    print_matrix("Attention Weights  softmax(masked scores)", attn_weights, tokens)

    out = attn_weights @ V
    print_matrix("Attention Output  (weights @ V)", out, tokens)
    return out


def transformer_block(x, Wq, Wk, Wv, W, Wff_up, Wff_down, block_name="Block"):
    print(f"\n{'█'*55}")
    print(f"  {block_name}")
    print(f"{'█'*55}")

    print(f"\n── Input to block ──")
    print_matrix("X", x, tokens)

    # 1. Attention sublayer
    print(f"\n┌─ Attention sublayer {'─'*30}")
    attn_out = swa_attention(x, Wq, Wk, Wv, W)

    # 2. Residual + LayerNorm
    x = F.layer_norm(x + attn_out, [x.shape[-1]])
    print(f"\n── After  attn + residual + LayerNorm ──")
    print_matrix("x", x, tokens)

    # 3. FFN sublayer: up-project -> ReLU -> down-project
    print(f"\n┌─ FFN sublayer {'─'*36}")
    ffn_out = F.relu(x @ Wff_up) @ Wff_down
    print_matrix("FFN output", ffn_out, tokens)

    # 4. Residual + LayerNorm
    x = F.layer_norm(x + ffn_out, [x.shape[-1]])
    print(f"\n── After  FFN + residual + LayerNorm  ← this is what enters the next block ──")
    print_matrix("x", x, tokens)

    return x


# ── Run ─────────────────────────────────────────────────────────
H1 = transformer_block(X,  Wq1, Wk1, Wv1, W, Wff1_up, Wff1_down, block_name="BLOCK 1")
H2 = transformer_block(H1, Wq2, Wk2, Wv2, W, Wff2_up, Wff2_down, block_name="BLOCK 2")



███████████████████████████████████████████████████████
  BLOCK 1
███████████████████████████████████████████████████████

── Input to block ──

── X  (shape [5, 3]) ──
       ['d0', 'd1', 'd2']
  A   [1.0, 0.0, 2.0]
  B   [0.0, 1.0, 1.0]
  C   [2.0, 1.0, 0.0]
  D   [1.0, 2.0, 1.0]
  E   [0.0, 0.0, 2.0]

┌─ Attention sublayer ──────────────────────────────

── Q  (X @ Wq)  (shape [5, 3]) ──
       ['d0', 'd1', 'd2']
  A   [1.0, 0.0, 2.0]
  B   [0.0, 1.0, 1.0]
  C   [2.0, 1.0, 0.0]
  D   [1.0, 2.0, 1.0]
  E   [0.0, 0.0, 2.0]

── K  (X @ Wk)  (shape [5, 3]) ──
       ['d0', 'd1', 'd2']
  A   [0.0, 1.0, 2.0]
  B   [1.0, 0.0, 1.0]
  C   [1.0, 2.0, 0.0]
  D   [2.0, 1.0, 1.0]
  E   [0.0, 0.0, 2.0]

── V  (X @ Wv)  (shape [5, 3]) ──
       ['d0', 'd1', 'd2']
  A   [1.0, 2.0, 0.0]
  B   [0.0, 1.0, 1.0]
  C   [2.0, 0.0, 1.0]
  D   [1.0, 1.0, 2.0]
  E   [0.0, 2.0, 0.0]

── SWA Mask  (W=2, causal) ──
  query↓  key→  ['A', 'B', 'C', 'D', 'E']
  A            ✓  ✗  ✗  ✗  ✗
  B            ✓  ✓  ✗  ✗

How SWA works in autoregressive decoding. The file is stored at: "e:\Ridwanur\Documents\Y3 FYM\toy_swa.py" Use terminal command:
$env:PYTHONIOENCODING='utf-8'; python toy_swa.py 


(base) PS E:\Ridwanur\Documents\Y3 FYM> $env:PYTHONIOENCODING='utf-8'; python toy_swa.py
╔═══════════════════════════════════════════════════════════════╗
║  SWA + GQA Autoregressive Decoding — KV Cache Simulation    ║
╠═══════════════════════════════════════════════════════════════╣
║  d=4, n_heads=2, n_kv_heads=1, head_dim=2, window_size=3           ║
║  Prompt: ['A', 'B', 'C']                                            ║
║  Decode: ['D', 'E']                                               ║
║  GQA ratio: 2 Q heads per KV head                             ║
╚═══════════════════════════════════════════════════════════════╝


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  PHASE 1: PREFILL  — process [A, B, C] in one shot
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

█████████████████████████████████████████████████████████████████
  PREFILL: tokens [A, B, C]
█████████████████████████████████████████████████████████████████

  Input shape: [1, 3, 4]  (B=1, S=3, d=4)
  The 4 columns of Q_full are 2 heads concatenated side by side
  ── Q_full  (x @ Wq)  [3, 4] ──
    A    [3.0, 2.0, 1.0, 2.0]
    B    [1.0, 2.0, 1.0, 0.0]
    C    [2.0, 1.0, 2.0, 3.0]

  ── K_new   (x @ Wk)  ← n_kv_heads×head_dim = 2  [3, 2] ──
    A    [3.0, 3.0]
    B    [1.0, 2.0]
    C    [2.0, 2.0]

  ── V_new   (x @ Wv)  [3, 2] ──
    A    [2.0, 4.0]
    B    [2.0, 1.0]
    C    [1.0, 3.0]

  Reshape into heads of the following shapes:
    Q: [1, 2, 3, 2]  (B, n_heads=2, S=3, head_dim=2)
    K: [1, 1, 3, 2]  (B, n_kv_heads=1, S=3, head_dim=2)

  ── No existing cache (first step) ──

  Cache size 3 ≤ window_size=3, no eviction needed

  ── KV Cache  (stored at n_kv_heads=1, 3 entries) ──
    Cached tokens: ['A', 'B', 'C']
    K cache shape: [1, 1, 3, 2]  (B, n_kv_heads, cache_len, head_dim)
    K[kv_head=0]:
      A: [3.0, 3.0]
      B: [1.0, 2.0]
      C: [2.0, 2.0]
    V[kv_head=0]:
      A: [2.0, 4.0]
      B: [2.0, 1.0]
      C: [1.0, 3.0]

  ── GQA Expand for attention ──
    K: [1, 1, 3, 2] → [1, 2, 3, 2]  (repeat for each KV head 2×)

  ── Attention Scores  Q @ Kᵀ / √head_dim  [1, 2, 3, 3] ──

    Head 0 (uses KV head 0):
      query↓  key→  ['A', 'B', 'C']
      A            [' 10.61', '  4.95', '  7.07']
      B            ['  6.36', '  3.54', '  4.24']
      C            ['  6.36', '  2.83', '  4.24']

    Head 1 (uses KV head 0):
      query↓  key→  ['A', 'B', 'C']
      A            ['  6.36', '  3.54', '  4.24']
      B            ['  2.12', '  0.71', '  1.41']
      C            [' 10.61', '  5.66', '  7.07']

  ── Causal + SWA Mask (window_size=3) ──
      query↓  key→  ['A', 'B', 'C']
      A             ✓  ✗  ✗
      B             ✓  ✓  ✗
      C             ✓  ✓  ✓

  ── Masked Scores ──

    Head 0:
      query↓  key→  ['A', 'B', 'C']
      A            [' 10.61', '  -inf', '  -inf']
      B            ['  6.36', '  3.54', '  -inf']
      C            ['  6.36', '  2.83', '  4.24']

    Head 1:
      query↓  key→  ['A', 'B', 'C']
      A            ['  6.36', '  -inf', '  -inf']
      B            ['  2.12', '  0.71', '  -inf']
      C            [' 10.61', '  5.66', '  7.07']

  ── Attention Weights (softmax) ──

    Head 0:
      query↓  key→  ['A', 'B', 'C']
      A            ['  1.00', '  0.00', '  0.00']
      B            ['  0.94', '  0.06', '  0.00']
      C            ['  0.87', '  0.03', '  0.10']

    Head 1:
      query↓  key→  ['A', 'B', 'C']
      A            ['  1.00', '  0.00', '  0.00']
      B            ['  0.80', '  0.20', '  0.00']
      C            ['  0.97', '  0.01', '  0.03']

  ── Per-Head Output (weights @ V)  [1, 2, 3, 2] ──
    Head 0:
      A: [2.0, 4.0]
      B: [2.0, 3.832578182220459]
      C: [1.8956730365753174, 3.819582462310791]
    Head 1:
      A: [2.0, 4.0]
      B: [2.0, 3.4132890701293945]
      C: [1.9718756675720215, 3.9513630867004395]

  ── Final Output (concat → Wo)  [1, 3, 4] ──
    A: [2.0, 4.0, 2.0, 4.0]
    B: [2.0, 3.83, 2.0, 3.41]
    C: [1.9, 3.82, 1.97, 3.95]


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  PHASE 2: AUTOREGRESSIVE DECODE  — one token at a time
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

█████████████████████████████████████████████████████████████████
  DECODE step 1: new token 'D' (position 3)
█████████████████████████████████████████████████████████████████

  Input shape: [1, 1, 4]  (B=1, S=1, d=4)

  ── Q_full  (x @ Wq)  [1, 4] ──
    D    [2.0, 3.0, 2.0, 1.0]

  ── K_new   (x @ Wk)  ← n_kv_heads×head_dim = 2  [1, 2] ──
    D    [2.0, 3.0]

  ── V_new   (x @ Wv)  [1, 2] ──
    D    [3.0, 2.0]

  Reshape into heads:
    Q: [1, 2, 1, 2]  (B, n_heads=2, S=1, head_dim=2)
    K: [1, 1, 1, 2]  (B, n_kv_heads=1, S=1, head_dim=2)

  ── Append to existing cache ──
    Past K shape:  [1, 1, 3, 2]  (3 cached tokens: ['A', 'B', 'C'])
    New K shape:   [1, 1, 1, 2]  (1 new token(s): ['D'])
    After cat:     [1, 1, 4, 2]  → tokens: ['A', 'B', 'C', 'D']

  ⚡ EVICTION: cache has 4 entries > window_size=3
    Evicting oldest 1 token(s): ['A']
    Cache after eviction: ['B', 'C', 'D']  (3 entries)

  ── KV Cache  (stored at n_kv_heads=1, 3 entries) ──
    Cached tokens: ['B', 'C', 'D']
    K cache shape: [1, 1, 3, 2]  (B, n_kv_heads, cache_len, head_dim)
    K[kv_head=0]:
      B: [1.0, 2.0]
      C: [2.0, 2.0]
      D: [2.0, 3.0]
    V[kv_head=0]:
      B: [2.0, 1.0]
      C: [1.0, 3.0]
      D: [3.0, 2.0]

  ── GQA Expand for attention ──
    K: [1, 1, 4, 2] → [1, 2, 4, 2]  (repeat each KV head 2×)

  ── Attention Scores  Q @ Kᵀ / √head_dim  [1, 2, 1, 4] ──

    Head 0 (uses KV head 0):
      query↓  key→  ['A', 'B', 'C', 'D']
      D            [' 10.61', '  5.66', '  7.07', '  9.19']

    Head 1 (uses KV head 0):
      query↓  key→  ['A', 'B', 'C', 'D']
      D            ['  6.36', '  2.83', '  4.24', '  4.95']

  ── Causal + SWA Mask (window_size=3) ──
      query↓  key→  ['A', 'B', 'C', 'D']
      D             ✗  ✓  ✓  ✓

  ── Masked Scores ──

    Head 0:
      query↓  key→  ['A', 'B', 'C', 'D']
      D            ['  -inf', '  5.66', '  7.07', '  9.19']

    Head 1:
      query↓  key→  ['A', 'B', 'C', 'D']
      D            ['  -inf', '  2.83', '  4.24', '  4.95']

  ── Attention Weights (softmax) ──

    Head 0:
      query↓  key→  ['A', 'B', 'C', 'D']
      D            ['  0.00', '  0.03', '  0.10', '  0.87']

    Head 1:
      query↓  key→  ['A', 'B', 'C', 'D']
      D            ['  0.00', '  0.07', '  0.31', '  0.62']

  ── Per-Head Output (weights @ V)  [1, 2, 1, 2] ──
    Head 0:
      D: [2.7659826278686523, 2.078963279724121]
    Head 1:
      D: [2.3142898082733154, 2.2313756942749023]

  ── Final Output (concat → Wo)  [1, 1, 4] ──
    D: [2.77, 2.08, 2.31, 2.23]

█████████████████████████████████████████████████████████████████
  DECODE step 2: new token 'E' (position 4)
█████████████████████████████████████████████████████████████████

  Input shape: [1, 1, 4]  (B=1, S=1, d=4)

  ── Q_full  (x @ Wq)  [1, 4] ──
    E    [2.0, 2.0, 1.0, 1.0]

  ── K_new   (x @ Wk)  ← n_kv_heads×head_dim = 2  [1, 2] ──
    E    [2.0, 3.0]

  ── V_new   (x @ Wv)  [1, 2] ──
    E    [2.0, 3.0]

  Reshape into heads:
    Q: [1, 2, 1, 2]  (B, n_heads=2, S=1, head_dim=2)
    K: [1, 1, 1, 2]  (B, n_kv_heads=1, S=1, head_dim=2)

  ── Append to existing cache ──
    Past K shape:  [1, 1, 3, 2]  (3 cached tokens: ['B', 'C', 'D'])
    New K shape:   [1, 1, 1, 2]  (1 new token(s): ['E'])
    After cat:     [1, 1, 4, 2]  → tokens: ['B', 'C', 'D', 'E']

  ⚡ EVICTION: cache has 4 entries > window_size=3
    Evicting oldest 1 token(s): ['B']
    Cache after eviction: ['C', 'D', 'E']  (3 entries)

  ── KV Cache  (stored at n_kv_heads=1, 3 entries) ──
    Cached tokens: ['C', 'D', 'E']
    K cache shape: [1, 1, 3, 2]  (B, n_kv_heads, cache_len, head_dim)
    K[kv_head=0]:
      C: [2.0, 2.0]
      D: [2.0, 3.0]
      E: [2.0, 3.0]
    V[kv_head=0]:
      C: [1.0, 3.0]
      D: [3.0, 2.0]
      E: [2.0, 3.0]

  ── GQA Expand for attention ──
    K: [1, 1, 4, 2] → [1, 2, 4, 2]  (repeat each KV head 2×)

  ── Attention Scores  Q @ Kᵀ / √head_dim  [1, 2, 1, 4] ──

    Head 0 (uses KV head 0):
      query↓  key→  ['B', 'C', 'D', 'E']
      E            ['  4.24', '  5.66', '  7.07', '  7.07']

    Head 1 (uses KV head 0):
      query↓  key→  ['B', 'C', 'D', 'E']
      E            ['  2.12', '  2.83', '  3.54', '  3.54']

  ── Causal + SWA Mask (window_size=3) ──
      query↓  key→  ['B', 'C', 'D', 'E']
      E             ✗  ✓  ✓  ✓

  ── Masked Scores ──

    Head 0:
      query↓  key→  ['B', 'C', 'D', 'E']
      E            ['  -inf', '  5.66', '  7.07', '  7.07']

    Head 1:
      query↓  key→  ['B', 'C', 'D', 'E']
      E            ['  -inf', '  2.83', '  3.54', '  3.54']

  ── Attention Weights (softmax) ──

    Head 0:
      query↓  key→  ['B', 'C', 'D', 'E']
      E            ['  0.00', '  0.11', '  0.45', '  0.45']

    Head 1:
      query↓  key→  ['B', 'C', 'D', 'E']
      E            ['  0.00', '  0.20', '  0.40', '  0.40']

  ── Per-Head Output (weights @ V)  [1, 2, 1, 2] ──
    Head 0:
      E: [2.3374247550964355, 2.5541915893554688]
    Head 1:
      E: [2.203336238861084, 2.5988879203796387]

  ── Final Output (concat → Wo)  [1, 1, 4] ──
    E: [2.34, 2.55, 2.2, 2.6]


═════════════════════════════════════════════════════════════════
  SUMMARY: KV Cache Evolution
═════════════════════════════════════════════════════════════════

  Step          | Action        | Cache before → after    | Evicted
  ──────────────┼───────────────┼─────────────────────────┼────────
  Prefill       | Insert A,B,C  | []       → [A, B, C]   | —
  Decode D      | Insert D      | [A,B,C]  → [A,B,C,D]   | A (len 4 > W=3)
                |               |          → [B, C, D]    |
  Decode E      | Insert E      | [B,C,D]  → [B,C,D,E]   | B (len 4 > W=3)
                |               |          → [C, D, E]    |

  Key insight: SWA layers only ever store 3 KV entries,
  regardless of sequence length. At position 1000, the cache is
  still just 3 entries — O(1) memory per layer.

  With GQA (n_kv_heads=1 vs n_heads=2), the cache
  is 2× smaller than standard MHA would require.